# Large Run Step 6: Additional Plotting Driver

Purpose: create optional diagnostic figures from existing compact tables and bounded samples, without loading full QC or metric inventories in the notebook.

Outputs: waveform comparison figures, station/event maps, flexible metric plots, and selected diagnostics.


## Setup
Purpose: load config/output paths and shared settings.

Outputs: printed paths and helper variables.


In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
repo_root = runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from IPython.display import display

from spatial_vtk.config import (
    notebook_run_context,
    notebook_figure_settings,
    prepare_notebook_geospatial_environment,
    print_notebook_context,
)
from spatial_vtk.io import output_group

prepare_notebook_geospatial_environment()
context = notebook_run_context()
cfg = context.cfg
repo_root = context.repo_root
config_path = context.config_path
outputs_root = context.outputs_root
tables_dir = context.tables_dir
figures_dir = context.figures_dir
slurm_dir = context.slurm_dir
logs_dir = context.logs_dir
RUN_LOCAL = context.run_local
SUBMIT_SLURM = context.submit_slurm
OVERWRITE = context.overwrite
QC_CHUNKSIZE = context.qc_chunksize
PREVIEW_ROWS = context.preview_rows

print_notebook_context(context)


## Resolve Plotting Inputs
Purpose: verify that comparison-eligible records and metric outputs are ready.

Outputs: status table only.


In [ ]:
step_outputs = output_group("step_06_plotting")

display(step_outputs.status_frame())


## Render Bounded Waveform Comparison
Purpose: load only a small comparison-eligible sample and render one waveform diagnostic figure.

Outputs: `event_trace_comparison` figure when `SVTK_MAKE_FIGURES=1`.


In [ ]:
WAVEFORM_FIGURE_SETTINGS = notebook_figure_settings("waveform", figure_dir=figures_dir)
waveform_figure_gate = WAVEFORM_FIGURE_SETTINGS.render_gate(
    [step_outputs.comparison_eligible_path, step_outputs.event_station_path],
    missing_message="Comparison-eligible records or event-station records are not ready yet.",
)
if not waveform_figure_gate.ready:
    print(waveform_figure_gate.message)
    display(waveform_figure_gate.status_frame())
else:
    from spatial_vtk.qc import build_qc_waveform_comparison_records, load_comparison_eligible_records
    from spatial_vtk.visualize.waveforms import plot_event_trace_comparison
    event_stations = step_outputs.load_table("event_station_records", cfg=context.cfg)
    sample = load_comparison_eligible_records(step_outputs.comparison_eligible_path, max_records=12, chunksize=QC_CHUNKSIZE)
    records = build_qc_waveform_comparison_records(event_stations, comparison_eligible=sample, max_records=12)
    plot_event_trace_comparison(records, savefig=True, **WAVEFORM_FIGURE_SETTINGS.plot_kwargs())
    print(f"Wrote {step_outputs.event_trace_comparison_path}")


## Preview Metric Plot Inputs
Purpose: inspect only the first rows of the metric table selected for plotting.

Outputs: bounded preview table.


In [ ]:
step_outputs.display_first_existing_table_preview(
    ("metrics_enriched_path", "metrics_long_path"),
    cfg=cfg,
    nrows=PREVIEW_ROWS,
    missing_message="Metric plotting source is not ready yet.",
)


## Region Boxplot with Comparison Table
Purpose: reproduce the tutorial region-comparison boxplot with its bootstrap comparison table using a bounded metric sample.


In [ ]:
REGION_FIGURE_SETTINGS = notebook_figure_settings(
    "region",
    figure_dir=figures_dir / "metrics",
    default_metric="PGA",
    default_passband="2-3 sec",
    default_sidecar_rows=1000,
)
region_figure_gate = REGION_FIGURE_SETTINGS.render_gate(
    [],
    disabled_message="Skipping region boxplot. Set SVTK_MAKE_FIGURES=1 to render it.",
)
if not region_figure_gate.ready:
    print(region_figure_gate.message)
    display(region_figure_gate.status_frame())
else:
    from spatial_vtk.spatial.plot import write_large_run_region_boxplot_from_outputs

    result = write_large_run_region_boxplot_from_outputs(
        step_outputs,
        figure_dir=figures_dir / "metrics",
        metric=REGION_FIGURE_SETTINGS.metric,
        passband=REGION_FIGURE_SETTINGS.passband,
        component=REGION_FIGURE_SETTINGS.component,
        model=REGION_FIGURE_SETTINGS.model,
        value_col=REGION_FIGURE_SETTINGS.value_col,
        compare_to=REGION_FIGURE_SETTINGS.compare_to,
        max_rows=REGION_FIGURE_SETTINGS.sample_rows,
        output_prefix="additional_region_boxplot",
        **REGION_FIGURE_SETTINGS.sidecars.kwargs(),
        annotate_if_missing=False,
        overwrite=OVERWRITE,
        showfig=REGION_FIGURE_SETTINGS.showfig,
    )
    print(result.message)
